In [ ]:
!git clone https://github.com/soCzech/TransNetV2.git


In [ ]:
import sys
import os
import cv2
import json
import glob
from PIL import Image
from tqdm import tqdm

# Thêm TransNetV2 vào đường dẫn hệ thống để import
sys.path.append('/kaggle/working/TransNetV2/inference')
from transnetv2 import TransNetV2

# 1. Định nghĩa đường dẫn
# Thay 'your-video-dataset' bằng tên dataset chứa video bạn up lên Kaggle
video_dir = '/kaggle/input/your-video-dataset/' # ĐƯỜNG DẪN MẪU
output_dir = '/kaggle/working/Keyframes'
os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(video_dir):
    print(f'THÔNG BÁO: Cần add dataset video vào {video_dir} qua nút "Add Input"')
    videos = []
else:
    videos = sorted(glob.glob(f"{video_dir}/**/*.mp4", recursive=True))
print(f"Tìm thấy {len(videos)} video.")

# 2. Khởi tạo model TransNetV2
if videos:
    print('Loading TransNetV2...')
    model = TransNetV2()

for video_path in tqdm(videos):
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    save_folder = os.path.join(output_dir, video_id)
    os.makedirs(save_folder, exist_ok=True)
    
    # Phát hiện cảnh (Scene Detection)
    _, single_frame_predictions, _ = model.predict_video(video_path)
    scenes = model.predictions_to_scenes(single_frame_predictions)
    
    # Trích xuất khung hình giữa của mỗi cảnh
    cap = cv2.VideoCapture(video_path)
    for i, scene in enumerate(scenes):
        start_frame, end_frame = scene
        middle_frame = int((start_frame + end_frame) // 2)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame)
        ret, frame_bgr = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame_rgb)
            img.save(f"{save_folder}/keyframe_{i:04d}.webp", "WEBP", lossless=True)
            
    cap.release()

print('Đã trích xuất xong keyframes vào thư mục /kaggle/working/Keyframes!')
